In [34]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from peft import PeftModelForCausalLM
from eval_length import load_gsm8k_parquet
from tqdm import tqdm
import copy

In [5]:
# model = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'
model = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-7B'
lora_path = '/workspace/25_11_17_r1qwen7b_grpo_budget_1_10_500_lr5e-4_alpha64_step27'

In [28]:
data = '/workspace/data/bigmath_1_10_500/test.parquet'
df = load_gsm8k_parquet(data)

In [46]:
tokenizer = AutoTokenizer.from_pretrained(model)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model on {device}...")

llm = AutoModelForCausalLM.from_pretrained(
    model,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
# llm = PeftModelForCausalLM.from_pretrained(llm, lora_path)
llm.eval()
# Create pipeline
pipe = pipeline(
    "text-generation",
    model=llm,
    tokenizer=tokenizer,
    device_map="auto",
)

Loading model on cuda...


Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.65s/it]
Device set to use cuda:0


In [29]:
df.iloc[1]['prompt']

array([{'content': 'We subtracted 50 from a certain number, divided it by 4, multiplied it by 3, and added 28. If the final result is 73, what is the certain number?\n\nYou have a token budget of around 500 tokens. You must finish your thinking process as close as possible to the thinking budget. Think step by step and output the final answer using \\boxed{}.', 'role': 'user'}],
      dtype=object)

In [41]:
inds = [1, 3, 5, 8]
prompt_messages = df.loc[inds, "prompt"].tolist()
prompt_messages

[array([{'content': 'We subtracted 50 from a certain number, divided it by 4, multiplied it by 3, and added 28. If the final result is 73, what is the certain number?\n\nYou have a token budget of around 500 tokens. You must finish your thinking process as close as possible to the thinking budget. Think step by step and output the final answer using \\boxed{}.', 'role': 'user'}],
       dtype=object),
 array([{'content': 'Kelsey travelled for 10 hours. He covered the first half of the distance at 25 kmph and the remaining half of the distance at a certain speed. Kelsey travelled a distance of 400 km. What was the speed at which Kelsey covered the remaining half of the distance?\n\nYou have a token budget of around 10 tokens. You must finish your thinking process as close as possible to the thinking budget. Think step by step and output the final answer using \\boxed{}.', 'role': 'user'}],
       dtype=object),
 array([{'content': 'A train is moving at a speed of 100 km/h and its length

In [42]:
for message in prompt_messages:
    message[0]['content'] = message[0]['content'] + '\n\nYou MUST NOT use the word "divide" in your reasoning. If you need to perform division, instead use some other expression or do that step silently. Remember, whatever you do, you MUST NOT use the word "divide".' 
prompt_messages

[array([{'content': 'We subtracted 50 from a certain number, divided it by 4, multiplied it by 3, and added 28. If the final result is 73, what is the certain number?\n\nYou have a token budget of around 500 tokens. You must finish your thinking process as close as possible to the thinking budget. Think step by step and output the final answer using \\boxed{}.\n\nYou MUST NOT use the word "divide" in your reasoning. If you need to perform division, instead use some other expression or do that step silently. Remember, whatever you do, you MUST NOT use the word "divide".', 'role': 'user'}],
       dtype=object),
 array([{'content': 'Kelsey travelled for 10 hours. He covered the first half of the distance at 25 kmph and the remaining half of the distance at a certain speed. Kelsey travelled a distance of 400 km. What was the speed at which Kelsey covered the remaining half of the distance?\n\nYou have a token budget of around 10 tokens. You must finish your thinking process as close as po

In [43]:
prompts = [
        tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        for messages in prompt_messages
    ]
prompts

['<｜begin▁of▁sentence｜><｜User｜>We subtracted 50 from a certain number, divided it by 4, multiplied it by 3, and added 28. If the final result is 73, what is the certain number?\n\nYou have a token budget of around 500 tokens. You must finish your thinking process as close as possible to the thinking budget. Think step by step and output the final answer using \\boxed{}.\n\nYou MUST NOT use the word "divide" in your reasoning. If you need to perform division, instead use some other expression or do that step silently. Remember, whatever you do, you MUST NOT use the word "divide".<｜Assistant｜><think>\n',
 '<｜begin▁of▁sentence｜><｜User｜>Kelsey travelled for 10 hours. He covered the first half of the distance at 25 kmph and the remaining half of the distance at a certain speed. Kelsey travelled a distance of 400 km. What was the speed at which Kelsey covered the remaining half of the distance?\n\nYou have a token budget of around 10 tokens. You must finish your thinking process as close as 

In [47]:
outputs = list(tqdm(
    pipe(
        (x for x in prompts),
        max_new_tokens=768,
        temperature=1,
        top_p=0.95,
        do_sample=True,
        num_return_sequences=1,
        pad_token_id=tokenizer.pad_token_id,
        return_full_text=False,
        batch_size=min(len(prompts), 256),
    ),
    total=len(prompts),
    desc="Generating responses"
))

Generating responses: 100%|██████████| 4/4 [00:13<00:00,  3.44s/it]


In [48]:
outputs

[[{'generated_text': "First, let's define the certain number as \\( x \\).\n\nAccording to the problem, we subtract 50 from \\( x \\), resulting in \\( x - 50 \\).\n\nNext, we divide this value by 4, which can be represented as \\( \\frac{x - 50}{4} \\).\n\nThen, we multiply by 3 to get \\( 3 \\times \\frac{x - 50}{4} \\).\n\nFinally, we add 28 to this product, leading to the equation:\n\\[\n3 \\times \\frac{x - 50}{4} + 28 = 73\n\\]\n\nTo solve for \\( x \\), we'll follow these steps:\n\n1. **Isolate the term with \\( x \\):**\n   Subtract 28 from both sides of the equation:\n   \\[\n   3 \\times \\frac{x - 50}{4} = 45\n   \\]\n\n2. **Simplify the equation:**\n   Multiply both sides by \\( \\frac{4}{3} \\) to eliminate the coefficient of \\( \\frac{x - 50}{4} \\):\n   \\[\n   x - 50 = 60\n   \\]\n\n3. **Solve for \\( x \\):**\n   Add 50 to both sides of the equation:\n   \\[\n   x = 110\n   \\]\n</think>\n\nLet's find the certain number step by step.\n\n**Step 1:** Let the certain num